# Part 1 - 실습 2: 자연어처리 기초 & 프롬프트 엔지니어링
**소요시간: 50분** | 난이도: ⭐⭐⭐

## 학습 목표
1. Zero-shot / Few-shot / Chain-of-Thought 프롬프팅 기법을 이해합니다.
2. System Prompt로 AI 페르소나와 출력 형식을 제어합니다.
3. Structured Output(JSON 응답) 기법을 실습합니다.

## 프롬프트 엔지니어링 핵심 기법
| 기법 | 설명 | 언제 사용 |
|---|---|---|
| **Zero-shot** | 예시 없이 지시만으로 태스크 수행 | 간단하고 명확한 작업 |
| **Few-shot** | 2~5개 예시를 함께 제공 | 특정 형식/스타일이 필요할 때 |
| **Chain-of-Thought** | "단계별로 생각하라" 지시 | 추론·계산이 필요한 복잡한 문제 |
| **System Prompt** | 역할·규칙·제약을 사전 정의 | 일관된 AI 행동 제어 |
| **Structured Output** | JSON 형식 응답 강제 | 프로그램에서 파싱할 때 |


---
## 🏢 기업 시나리오 — 추가 개발 없이 업무 자동화

당신은 **데이터 분석가**입니다. 모델을 학습시키지 않고, **프롬프트만 잘 써서** 업무를 자동화합니다.

이번 실습에서는 다음을 익힙니다.
1. **Zero/Few-shot** → 리뷰·문의 자동 분류
2. **Chain-of-Thought** → 복잡한 계산·심사 로직을 단계별로
3. **Structured Output(JSON)** → 시스템에 바로 넣을 수 있는 데이터로 추출
4. **System Prompt** → 일관된 회사 톤·페르소나 제어

> 💡 실무 핵심은 **Structured Output**입니다. 응답을 JSON으로 받으면 그대로 DB·API에 연결됩니다. 예: 고객 문의 → {유형, 긴급도, 담당부서} 자동 태깅.


In [ ]:
# ✅ [제공 코드] 환경 초기화 & 헬퍼 함수
import boto3, json

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

def claude(user_prompt, system_prompt=None, temperature=0.7, max_tokens=1024):
    """Claude converse API 래퍼"""
    kwargs = {
        'modelId': MODEL_ID,
        'messages': [{'role': 'user', 'content': [{'text': user_prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]
    resp = bedrock_runtime.converse(**kwargs)
    return resp['output']['message']['content'][0]['text']

print('✅ claude() 헬퍼 함수 준비 완료')


## ✏️ TODO 1: Zero-shot vs Few-shot 비교

동일한 감성 분류 태스크를 Zero-shot과 Few-shot으로 각각 수행하고 결과를 비교하세요.


In [ ]:
# ✏️ TODO 1: Zero-shot vs Few-shot 프롬프트를 작성하고 결과를 비교하세요
test_reviews = [
    '이 강의 정말 유익했어요. 내용도 알차고 강사님도 친절하세요.',
    '과제가 너무 많고 설명이 부족해서 따라가기 힘들었습니다.',
    '그냥 평범한 수업이에요. 특별히 좋거나 나쁘지 않았습니다.',
]

# Zero-shot 프롬프트
zero_shot_template = """
다음 리뷰의 감성을 분석하여 '긍정', '부정', '중립' 중 하나만 답하세요.
리뷰: {review}
"""

# Few-shot 프롬프트 (예시 포함)
few_shot_template = """
다음은 강의 리뷰 감성 분류 예시입니다.

예시:
리뷰: '최고의 강의입니다! 강력 추천해요.'  → 긍정
리뷰: '돈이 아까웠습니다. 전혀 도움 안 됨.'  → 부정
리뷰: '수업 내용은 보통이었어요.'  → 중립

이제 다음 리뷰를 분류하세요.
리뷰: '{review}'  →
"""

# 라벨만 추출하는 헬퍼 (모델이 설명을 덧붙여도 표가 깨지지 않도록)
def to_label(text):
    for label in ('긍정', '부정', '중립'):
        if label in text:
            return label
    return text.replace('\n', ' ')[:20]

# 마크다운 표로 출력 — 셀 출력을 복사해 마크다운 에디터에 바로 붙여넣을 수 있습니다
rows = ['| 리뷰 | Zero-shot | Few-shot |', '|---|:---:|:---:|']
for review in test_reviews:
    # Zero-shot 호출
    zero_result = claude(
        zero_shot_template.format(_____=_____),   # ← review=review
        temperature=_____                          # ← 0.1 (일관된 분류를 위해 낮게)
    ).strip()

    # Few-shot 호출
    few_result = claude(
        few_shot_template.format(_____=_____),    # ← review=review
        temperature=_____                          # ← 0.1
    ).strip()

    rows.append(f'| {review} | {to_label(zero_result)} | {to_label(few_result)} |')

markdown_table = '\n'.join(rows)
print(markdown_table)                # ← 이 텍스트를 복사해서 사용

# 노트북 안에서 렌더링 결과도 바로 확인
from IPython.display import Markdown, display
display(Markdown(markdown_table))


## ✏️ TODO 2: Chain-of-Thought 프롬프팅

복잡한 문제를 단계별로 추론하도록 Chain-of-Thought 프롬프트를 작성하세요.


In [ ]:
# ✏️ TODO 2: CoT 프롬프트를 작성하여 단계별 추론을 유도하세요
problem = """
학교 AI 강의 수강생이 20명이다.
첫 번째 과제를 제출한 학생은 전체의 80%이다.
그 중 60%가 두 번째 과제도 제출했다.
두 번째 과제를 제출하지 않은 학생은 몇 명인가?
"""

# 일반 프롬프트 (CoT 없음)
normal_prompt = f'다음 문제를 풀어주세요: {problem}'

# CoT 프롬프트 — 빈칸 완성
cot_prompt = f"""
다음 문제를 단계별로 풀어주세요.

규칙:
1. 각 계산 단계를 [단계 N] 형식으로 표시하세요.
2. 각 단계에서 수식과 결과를 명시하세요.
3. 마지막에 [최종 답변]을 반드시 제시하세요.

문제: {_____}
"""

print('=== 일반 프롬프트 결과 ===')
print(claude(normal_prompt, temperature=0.1))
print()
print('=== Chain-of-Thought 결과 ===')
print(claude(_____, temperature=0.1))   # ← cot_prompt


## ✏️ TODO 3: Structured Output — JSON 응답 강제

AI가 항상 파싱 가능한 JSON으로 응답하도록 프롬프트를 설계하세요.


In [ ]:
# ✏️ TODO 3: JSON 형식 응답을 강제하는 System Prompt를 작성하세요
SYSTEM_JSON = """
당신은 텍스트 분석 AI입니다.
반드시 다음 JSON 형식으로만 응답하세요. JSON 외 다른 텍스트는 절대 포함하지 마세요.

{
  "sentiment": "긍정|부정|중립",
  "confidence": 0.0~1.0,
  "keywords": ["키워드1", "키워드2", "키워드3"],
  "summary": "한 문장 요약"
}
"""

texts = [
    '오늘 딥러닝 강의를 들었는데 내용이 너무 어려워서 힘들었지만 끝내고 나니 뿌듯했다.',
    '과제 제출 마감이 너무 촉박해서 밤을 새웠다. 다음엔 여유 있게 시작해야겠다.',
]

for text in texts:
    raw = claude(
        user_prompt=_____,          # ← text
        system_prompt=_____,        # ← SYSTEM_JSON
        temperature=_____           # ← 0.1 (JSON 일관성을 위해)
    )
    try:
        parsed = json.loads(_____) # ← raw
        print(f"텍스트: {text[:30]}...")
        print(f"  감성: {parsed[_____]}  신뢰도: {parsed['confidence']}")   # ← 'sentiment'
        print(f"  키워드: {parsed['keywords']}")
        print(f"  요약: {parsed['summary']}\n")
    except json.JSONDecodeError:
        print(f'JSON 파싱 실패:\n{raw}')


## ✏️ TODO 4: System Prompt로 페르소나 제어

학교 AI 강의 조교 페르소나를 System Prompt로 정의하고, 동일 질문에 대해 다른 페르소나와 응답을 비교하세요.


In [ ]:
# ✏️ TODO 4: 두 가지 System Prompt 페르소나를 정의하고 응답을 비교하세요
SYSTEM_TUTOR = """
당신은 대학교 AI 강의 조교입니다.
다음 규칙을 반드시 지키세요:
- 친절하고 격려하는 톤으로 답변하세요.
- 개념 설명 시 현실 예시를 반드시 포함하세요.
- 학생이 이해하기 어렵다고 하면 더 쉬운 비유를 사용하세요.
- 답변 끝에 관련 심화 학습 질문을 하나 제안하세요.
"""

SYSTEM_EXPERT = """
당신은 AI 연구소 선임 연구원입니다.
다음 규칙을 반드시 지키세요:
- 기술적으로 정확하고 간결하게 답변하세요.
- 전문 용어를 사용하되 필요시 영문 원어를 병기하세요.
- 학술 논문 또는 공식 문서 기준으로 설명하세요.
"""

question = 'Transformer 모델의 Attention 메커니즘이 무엇인가요?'

print('=== 강의 조교 페르소나 ===')
print(claude(question, system_prompt=_____))   # ← SYSTEM_TUTOR
print()
print('=== 연구원 페르소나 ===')
print(claude(question, system_prompt=_____))   # ← SYSTEM_EXPERT


---
## 🔗 실무로 연결하기

프롬프트 엔지니어링은 **'모델 학습 없이' 업무를 자동화**하는 가장 빠른 길입니다.

- **Structured Output(JSON)**: 고객 문의 → `{유형, 긴급도, 부서}` 로 자동 분류해 티켓 시스템에 바로 연결
- **Few-shot**: 회사만의 분류 기준을 예시 몇 개로 학습 없이 주입
- **System Prompt**: 모든 답변을 회사 톤·정책에 맞게 일관 유지


## 💡 심화 도전
1. 동일 질문을 Zero-shot, Few-shot, CoT 세 가지 방식으로 호출해 토큰 수(응답 길이)를 비교해보세요.
2. JSON 응답에 `sentiment_reason` 필드를 추가하도록 System Prompt를 수정하세요.
3. 학생이 '모르겠어요'라고 하면 힌트만 주는 소크라테스식 교육 페르소나를 설계해보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1
zero_result = claude(zero_shot_template.format(review=review), temperature=0.1)
few_result  = claude(few_shot_template.format(review=review),  temperature=0.1)

# TODO 2
cot_prompt = f'...문제: {problem}...'
claude(cot_prompt, temperature=0.1)

# TODO 3
raw    = claude(user_prompt=text, system_prompt=SYSTEM_JSON, temperature=0.1)
parsed = json.loads(raw)
parsed['sentiment']

# TODO 4
claude(question, system_prompt=SYSTEM_TUTOR)
claude(question, system_prompt=SYSTEM_EXPERT)
```
